### DAPO Train Files / Val Files

In [ ]:
from datasets import load_dataset
from pprint import pprint
from collections import defaultdict
from tqdm import tqdm

path = "/home/ma-user/work/dev/_huggingface/_datasets/_psrnsr/DAPO-Math-17K"
dataset = load_dataset(path)['train']

In [27]:
SYS_PROMPT="Please reason step by step, and put your final answer within \\boxed{}."
print(SYS_PROMPT)

Please reason step by step, and put your final answer within \boxed{}.


In [39]:
PREFIX="Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\n"
SUFFIX='\n\nRemember to put your answer on its own line after "Answer:".'

sample = dataset[1]['prompt'][0]['content']

print(sample.startswith(PREFIX))
print(sample.endswith(SUFFIX))
print(sample.lstrip(PREFIX).rstrip(SUFFIX))

True
True
Let $ABCD$ be a unit square in the plane. Points $X$ and $Y$ are chosen independently and uniformly at random on the perimeter of $ABCD$. If the expected value of the area of triangle $\triangle AXY$ can be expressed as $\frac{m}{n}$ for relatively prime positive integers $m$ and $n$, compute $m+n$


In [40]:
dataset[0]

{'data_source': 'math_dapo',
 'prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.\n\nRemember to put your answer on its own line after "Answer:".',
   'role': 'user'}],
 'ability': 'MATH',
 'reward_model': {'ground_truth': '34', 'style': 'rule-lighteval/MATH_v2'},
 'extra_info': {'index': '9a9b6eb4-a1cb-49d1-8c1e-62eaf2f74079'}}

In [49]:
from datasets import Dataset

# 假设你的变量叫 dataset
seen_ids = set()
keep_indices = []

# 遍历数据集，记录第一次出现的 index
for i, item in enumerate(dataset):
    # 提取唯一标识
    uid = item['extra_info']['index']
    
    if uid not in seen_ids:
        seen_ids.add(uid)
        keep_indices.append(i)

# 使用 select 只保留唯一的那些行
deduped_dataset = dataset.select(keep_indices)

print(f"原始数量: {len(dataset)}")
print(f"去重后数量: {len(deduped_dataset)}")

原始数量: 1791700
去重后数量: 17917


In [51]:
print(len(deduped_dataset))

17917


In [74]:
def map_func(line):
    prompt = line['prompt']
    p = prompt[0]['content']
    assert p.startswith(PREFIX)
    assert p.endswith(SUFFIX)

    p = p.lstrip(PREFIX).rstrip(SUFFIX)
    
    p = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": p}
    ]

    return {
        "prompt": p
    }

In [75]:
final_dataset = deduped_dataset.map(map_func)

Map:   0%|          | 0/17917 [00:00<?, ? examples/s]

In [76]:
print(final_dataset[0]['prompt'])

[{'content': 'Please reason step by step, and put your final answer within \\boxed{}.', 'role': 'system'}, {'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$', 'role': 'user'}]


In [77]:
save_path = "/home/ma-user/work/dev/_huggingface/_datasets/_psrnsr/DAPO-Math-17K/dapo1223_17k.parquet"

final_dataset.to_parquet(save_path)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

8237862

## ValSet 构建

In [68]:
path = "/home/ma-user/work/dev/_huggingface/_datasets/_data2/processed_dataset_new/luffy_test_sys_wo_format/test.parquet"

val = load_dataset("parquet", data_files=path)['train']

In [72]:
from collections import defaultdict

counts = defaultdict(int)

for line in val:
    s = line['data_source']
    counts[s] += 1

for k,v in counts.items():
    print(k, v)

math 500
olympiad_bench 675
minerva 272
aime 960
amc 2656
aime25 960


In [78]:
SYS_PROMPT="Please reason step by step, and put your final answer within \\boxed{}."

In [81]:
def map_func(line):
    prompt = line['prompt']
    
    query = prompt[1]['content']

    
    p = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": query}
    ]

    return {
        "prompt": p
    }

In [82]:
val = val.map(map_func)

Map:   0%|          | 0/6023 [00:00<?, ? examples/s]

In [85]:
print(val[0])

{'data_source': 'math', 'prompt': [{'content': 'Please reason step by step, and put your final answer within \\boxed{}.', 'role': 'system'}, {'content': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '\\left( 3, \\frac{\\pi}{2} \\right)', 'style': 'rule'}, 'extra_info': {'index': 0, 'split': 'default'}, '__index_level_0__': 240}


In [86]:
val.to_parquet("/home/ma-user/work/dev/_huggingface/_datasets/_psrnsr/DAPO-Math-17K/val_full.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2867868

In [ ]:
bmks = ["amc", "aime24", "olympiad_bench"]